In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import re

In [2]:
time_df = pd.read_csv("testbench_results.csv")
size_df = pd.read_csv("testbench_size_results.csv")

In [3]:
keys = [
    "FILE_NAME",
    "MAX_HEIGHT",
    "PHRASE_NUM"]

size_df_reduced = (size_df.groupby(keys, as_index=False)["BYTES"].mean())

merged_df = time_df.merge(
    size_df_reduced,
    on=keys,
    how="left"
)

merged_df["BYTES"].isna().sum()
check = (
    size_df
    .groupby(keys)["BYTES"]
    .nunique()
)

print(check[check > 1])


Series([], Name: BYTES, dtype: int64)


In [4]:
df = merged_df

# Extract algorithm from file path
df["PATH"] = df["FILE_NAME"].astype(str)
df["ALGORITHM"] = df["PATH"].apply(
    lambda p: os.path.normpath(p).split(os.sep)[-2]
)

df["FILENAME"] = df["PATH"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
)


# Extract dataset from file path
def extract_dataset(name):
    if "_h" in name:
        return name.split("_h")[0].split("/")[-1].split(".")[0]
    if "_lz-" in name:
        return name.split("_lz-")[0].split("/")[-1].split(".")[0]
    return name.split("/")[-1].split(".")[0]

# /home/timmo/Code/lzhb-testbench/res/c4/influenza_h5_a0.1_b0.1_g0_c4.lzcp
df["ALPHA"] = df["FILE_NAME"].apply(lambda n: re.search(r"_a([\d.]+)", n).group(1) if re.search(r"_a([\d.]+)", n) else None)
df["BETA"] = df["FILE_NAME"].apply(lambda n: re.search(r"_b([\d.]+)", n).group(1) if re.search(r"_b([\d.]+)", n) else None)

df["DATASET"] = df["FILE_NAME"].apply(extract_dataset)


def extract_height(name):
    m = re.search(r"_h(\d+)", name)
    return int(m.group(1)) if m else None

df["HEIGHT_BOUND"] = df["FILE_NAME"].apply(extract_height)
df["IS_HEIGHT_BOUND"] = df["HEIGHT_BOUND"].notna()

df[["FILE_NAME", "DATASET"]].head(10)




,FILE_NAME,DATASET
0,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
1,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
2,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
3,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
4,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
5,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
6,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
7,/home/timmo/Code/lzhb-testbench/res/c1/coreuti...,coreutils
8,/home/timmo/Code/lzhb-testbench/res/c1/einstei...,einstein
9,/home/timmo/Code/lzhb-testbench/res/c1/einstei...,einstein


In [5]:
extract_dataset("/home/timmo/Code/lzhb-testbench/res/kpp3/coreutils.lzcp")
#extract_dataset("/home/timmo/Code/lzhb-testbench/res/lzlmocc/coreutils_lz-lmocc.lzcp")
df[df["ALGORITHM"] == "kpp3"]

,TIMESTAMP,FILE_NAME,REPEATS,BATCH_SIZE,PHRASE_NUM,MAX_HEIGHT,AVG_HEIGHT,VAR_HEIGHT,MAX_LENGTH,AVG_LENGTH,...,AVERAGE_STRING_CONSEC_CHAR_TIME_NS,BYTES,PATH,ALGORITHM,FILENAME,ALPHA,BETA,DATASET,HEIGHT_BOUND,IS_HEIGHT_BOUND


In [6]:
DATASET = "coreutils"

df_ds = df[df["DATASET"] == DATASET]
df_flat = df_ds[~df_ds["IS_HEIGHT_BOUND"]]
df_flat[["ALGORITHM", "BYTES", "AVERAGE_ACCESS_CHAR_TIME"]]

df_not_flat = df_ds[df_ds["IS_HEIGHT_BOUND"]]

In [13]:
def summarize_height_group(df_group, space_percentile=0.1):
    """
    df_group: subset of df_not_flat for a particular algorithm or dataset
    space_percentile: fraction (0.1 = 10%) of minimum space to filter
    Returns: two rows
    """
    # Absolute minimum average access time
    idx_min_time = df_group["AVERAGE_ACCESS_CHAR_TIME"].idxmin()
    row_min_time = df_group.loc[idx_min_time]

    # Minimum average access time among small-space subset
    min_space = df_group["BYTES"].min()
    space_threshold = min_space * (1 + space_percentile)  # within percentile of min
    small_space_df = df_group[df_group["BYTES"] <= space_threshold]

    idx_min_time_small_space = small_space_df["AVERAGE_ACCESS_CHAR_TIME"].idxmin()
    row_min_time_small_space = df_group.loc[idx_min_time_small_space]

    return pd.DataFrame([row_min_time, row_min_time_small_space])
  
  

df_hb = df_not_flat.copy()
# Group by algorithm and dataset (or category if you have one)
summary_hb = df_hb.groupby("ALGORITHM").apply(lambda g: summarize_height_group(g)).reset_index(drop=False)


df_flat["KB"] = df_flat["BYTES"] / 1024
summary_hb["KB"] = summary_hb["BYTES"] / 1024

df_flat["MB"] = df_flat["BYTES"] / (1024 * 1024)
summary_hb["MB"] = summary_hb["BYTES"] / (1024 * 1024)

In [14]:
import plotly.express as px
import matplotlib.cm as cm

# --- Assign colors per algorithm ---
all_algorithms = pd.concat([df_flat["ALGORITHM"], summary_hb["ALGORITHM"]]).unique()
cmap = cm.get_cmap("tab10", len(all_algorithms))
def mpl_color_to_rgb_str(color):
    r, g, b, *_ = color
    return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
algo_color_map = {algo: mpl_color_to_rgb_str(cmap(i)) for i, algo in enumerate(all_algorithms)}

# --- Flat data ---
df_flat_plot = df_flat.copy()
df_flat_plot["LEGEND_LABEL"] = df_flat_plot["ALGORITHM"] + " (flat)"
df_flat_plot["MARKER"] = "circle"
df_flat_plot["COLOR_ALGO"] = df_flat_plot["ALGORITHM"].map(algo_color_map)

# --- Height-bounded data ---
df_hb_plot = summary_hb.copy().reset_index(drop=True)
markers = ["x", "triangle-up"]
labels = ["lowest avg", "lowest avg in small space"]

df_hb_plot["LEGEND_LABEL"] = [
    f"{row.ALGORITHM} ({labels[i % 2]})"
    for i, row in enumerate(df_hb_plot.itertuples())
]
df_hb_plot["MARKER"] = [markers[i % 2] for i in range(len(df_hb_plot))]
df_hb_plot["COLOR_ALGO"] = df_hb_plot["ALGORITHM"].map(algo_color_map)

# --- Combine ---
df_plot = pd.concat([df_flat_plot, df_hb_plot], ignore_index=True)

# --- Plot ---
fig = px.scatter(
    df_plot,
    x="MB",
    y="AVERAGE_ACCESS_CHAR_TIME",
    color="LEGEND_LABEL",          # legend shows descriptive label
    symbol="MARKER",               # marker shape distinguishes type
    hover_data=["ALGORITHM", "MB", "AVERAGE_ACCESS_CHAR_TIME", "HEIGHT_BOUND", "ALPHA", "BETA"],
    color_discrete_map={label: row.COLOR_ALGO for label, row in df_plot.set_index("LEGEND_LABEL").iterrows()},
    log_x=False,
    log_y=False,
    labels={
        "MB": "Space (MB) [Phrases + Predecessor Table]",
        "AVERAGE_ACCESS_CHAR_TIME": "Average character access time (ns)"
    },
    title=f"{DATASET}: Space vs Access Time | 100.000 Batches of 512 Random Access of Characters"
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='black')))


fig.update_layout(legend_title_text='Algorithm / Variant')
fig.write_html(f"{DATASET}_plot.html", include_plotlyjs='cdn')
fig.show()



/tmp/ipykernel_91884/3090326598.py:6: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

